# Bar cache explorer

Audits a cache under `data/` and runs the buy-and-hold vs. timing comparison
on whatever is present.

Fill the cache first (needs TWS open):

```bash
python examples/fetch_bar_cache.py data5y --etfs --duration "5 Y"
```

**Clear outputs before committing** — executed cells embed account figures
and megabytes of plot data in the diff:

```bash
jupyter nbconvert --clear-output --inplace notebooks/*.ipynb
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `import paths` works

import pandas as pd

from ibkr.cache import cached_symbols, load_cache
from paths import bar_cache_dir
from strategy.data_quality import drop_incomplete_trailing_bar

CACHE = "data5y"  # or "data2y"
cache = bar_cache_dir(CACHE)
print(cache, "->", len(cached_symbols(cache)), "symbols cached")

## Load, dropping any incomplete trailing bar

A mid-session bar cached and reused after the close produced a false gate
failure on 2026-08-03 — that is what `drop_incomplete_trailing_bar` guards.
Always check `skipped`: `load_cache` skips bad files rather than raising, so
ignoring it means silently working on a smaller universe than you think.

In [ ]:
frames, skipped = load_cache(
    cache,
    min_bars=250,
    transform=lambda df: drop_incomplete_trailing_bar(df)[0],
)

print(f"loaded {len(frames)}, skipped {len(skipped)}")
for sym, why in skipped:
    print(f"  {sym:<6} {why}")

## Coverage audit

Uneven start dates are usually legitimate (SNDK and WOLF have genuinely
short history from a spinoff/reorg). A *ragged* file is the corrupt case and
never reaches this table — `load_cache` rejects it into `skipped`.

In [ ]:
coverage = pd.DataFrame(
    [
        {
            "symbol": sym,
            "bars": len(df),
            "start": df.index[0].date(),
            "end": df.index[-1].date(),
            "last_close": round(float(df["close"].iloc[-1]), 2),
        }
        for sym, df in frames.items()
    ]
).sort_values("bars")

coverage.head(15)

In [ ]:
SYMBOL = "QQQ" if "QQQ" in frames else sorted(frames)[0]
df = frames[SYMBOL]

ax = df["close"].plot(figsize=(12, 5), title=f"{SYMBOL} close", lw=1)
df["close"].rolling(200).mean().plot(ax=ax, lw=1, label="200-day SMA")
ax.legend()

peak = df["close"].cummax()
print(f"max drawdown over the window: {((df['close'] / peak) - 1).min():.1%}")

## Buy-and-hold vs. timing, net of real commissions

The commission model is this account's measured schedule —
`clamp($0.005/share, min $1.00, max 1% of trade value)`. Gross numbers were
never the problem; at ~$35 positions the 1% cap binds both ways, so every
round trip costs ~2% of position value. Any strategy compared here without
that applied is not being compared honestly.

In [ ]:
from backtest.lowfreq import run_lowfreq
from backtest.strategies_lowfreq import buy_and_hold, sma_timing

CAPITAL = 300.43
rows = []
for sym in (s for s in ("QQQ", "SPY") if s in frames):
    one = {sym: frames[sym]}
    rows.append(run_lowfreq(f"{sym} buy & hold", one, buy_and_hold(sym), CAPITAL, freq="never").summary())
    rows.append(run_lowfreq(f"{sym} 200d timing", one, sma_timing(sym), CAPITAL, freq="M").summary())

pd.DataFrame(rows)[
    ["name", "total_return_pct", "cagr_pct", "max_drawdown_pct", "fills", "commission", "end"]
].sort_values("cagr_pct", ascending=False)

Expect buy-and-hold to win on return and timing to win on drawdown — that is
the result the account's strategy was chosen from, and it held across every
active overlay tested. Read `docs/backtest-verdict.md` for the caveats
(a five-year window starting *after* the 2022 bear, unadjusted prices) before
concluding anything from a single table here.